# 인공지능응용 · Week 10 · 이미지 분류의 학습과 평가 분리하기

**본인 사본을 만들어 실행·수정·기록하세요.**

- 이름: **⟦여기에 직접 입력⟧**
- 학번: **⟦여기에 직접 입력⟧**

[자습 자료](https://chorok-daddy.github.io/courses/ai-applications/image-classification/index.html) · [실습 안내](https://chorok-daddy.github.io/courses/ai-applications/image-classification/assignment.html)

## 시작 전 · 셀 실행과 작성 방법
Code Cell 왼쪽 ▶ 또는 Shift+Enter로 실행하세요. 위에서부터 진행하며 앞 셀의 변수를 사용합니다. Text Cell은 더블클릭해 **⟦직접 입력⟧** 부분을 바꾸고 Shift+Enter로 표시합니다. 기준 예제는 실행해서 이해하고, **직접 작성** 셀에 본인 코드를 작성하세요. 값을 바꾸면 해당 셀과 뒤의 관련 셀을 다시 실행합니다.

**60분 진행:** 기준 예제 10분 → 조건 변경·반복 30분 → 오류 수정·해석 15분 → 저장·점검 5분. 시간이 남으면 마지막 선택 실습을 수행하세요.

Colab에 필요한 라이브러리가 없으면 새 런타임에서 환경을 확인하고 조교에게 문의하세요. CPU로 기준 실습을 실행할 수 있습니다. 외부 다운로드가 필요한 실습은 해당 셀에 표시합니다.

## 1. MNIST와 실행 규모

처음에는 빠른 실행으로 연결을 확인합니다. 전체 실습에서는 QUICK_RUN=False로 바꾸고 실제 사용 표본 수를 보고하세요.

### A. 기준 예제 · 먼저 읽고 실행

In [ ]:
import sys, numpy as np, matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
torch.manual_seed(42)
torch.set_num_threads(2)
print('Python:', sys.version.split()[0], '| PyTorch:', torch.__version__)
import torchvision
from torchvision import datasets,transforms
from torch.utils.data import DataLoader,Subset
QUICK_RUN=True # 직접 변경: 전체 데이터는 False
train_data=datasets.MNIST('mnist_data',train=True,download=True,transform=transforms.ToTensor())
test_data=datasets.MNIST('mnist_data',train=False,download=True,transform=transforms.ToTensor())
train_used=Subset(train_data,range(1200)) if QUICK_RUN else train_data
test_used=Subset(test_data,range(300)) if QUICK_RUN else test_data
train_loader=DataLoader(train_used,batch_size=64,shuffle=True,generator=torch.Generator().manual_seed(42))
test_loader=DataLoader(test_used,batch_size=128,shuffle=False)
print('Mode:', 'QUICK' if QUICK_RUN else 'FULL','train:',len(train_used),'test:',len(test_used))
img,label=train_data[0];print(img.shape,label);plt.imshow(img.squeeze(),cmap='gray');plt.show()



### B. 직접 수행

이미지 한 장의 shape와 label을 확인하고 DataLoader의 마지막 Batch(배치) 크기를 조사하세요.

먼저 결과를 예상하고 코드를 작성하세요. 위 예제의 결과만으로 이 항목을 완료한 것은 아닙니다.

In [ ]:
# ✍ 직접 작성: 이 셀 아래에 본인의 코드를 추가하세요.
# 코드가 길어지면 Code Cell을 추가해도 됩니다.


### C. 직접 기록 · 이 Text Cell을 더블클릭해서 수정

- 바꾼 조건: **⟦직접 입력⟧**
- 실행 전 예상: **⟦직접 입력⟧**
- 실제 출력/그래프에서 확인한 값: **⟦직접 입력⟧**
- 예상과 차이 및 설명: **⟦직접 입력⟧**

## 2. CNN 학습과 평가

평가에서는 파라미터를 갱신하지 않습니다.

### A. 기준 예제 · 먼저 읽고 실행

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1=nn.Sequential(nn.Conv2d(1,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.layer2=nn.Sequential(nn.Conv2d(32,64,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.layer3=nn.Sequential(nn.Conv2d(64,128,3,padding=1),nn.ReLU(),nn.MaxPool2d(2,padding=1))
        self.fc1=nn.Linear(128*4*4,625);self.relu=nn.ReLU();self.dropout=nn.Dropout();self.fc2=nn.Linear(625,10)
    def forward(self,x):
        x=self.layer3(self.layer2(self.layer1(x)));x=x.flatten(1)
        return self.fc2(self.dropout(self.relu(self.fc1(x))))
model=CNN();optimizer=torch.optim.Adam(model.parameters(),lr=.001)
epochs=2 if QUICK_RUN else 5
loss_history=[];accuracy_history=[]
for epoch in range(epochs):
    model.train();total_loss=0.;correct=0;count=0
    for xb,yb in train_loader:
        logits=model(xb);loss=F.cross_entropy(logits,yb)
        optimizer.zero_grad();loss.backward();optimizer.step()
        total_loss+=loss.item()*len(yb);correct+=(logits.argmax(1)==yb).sum().item();count+=len(yb)
    loss_history.append(total_loss/count);accuracy_history.append(correct/count)
    print(epoch+1,loss_history[-1],accuracy_history[-1])
model.eval();correct=0;count=0;examples=[];mistakes=[]
with torch.no_grad():
    for xb,yb in test_loader:
        pred=model(xb).argmax(1);correct+=(pred==yb).sum().item();count+=len(yb)
        for image,truth,guess in zip(xb,yb,pred):
            item=(image.squeeze(),truth.item(),guess.item())
            if len(examples)<10:examples.append(item)
            if truth!=guess and len(mistakes)<5:mistakes.append(item)
print('TEST:',correct/count,'samples:',count)



### B. 직접 수행

전체 데이터로 실행한 경우 사용 epoch와 학습·평가 데이터 수를 기록하세요. train과 eval, no_grad의 서로 다른 역할을 설명하세요.

먼저 결과를 예상하고 코드를 작성하세요. 위 예제의 결과만으로 이 항목을 완료한 것은 아닙니다.

In [ ]:
# ✍ 직접 작성: 이 셀 아래에 본인의 코드를 추가하세요.
# 코드가 길어지면 Code Cell을 추가해도 됩니다.


### C. 직접 기록 · 이 Text Cell을 더블클릭해서 수정

- 바꾼 조건: **⟦직접 입력⟧**
- 실행 전 예상: **⟦직접 입력⟧**
- 실제 출력/그래프에서 확인한 값: **⟦직접 입력⟧**
- 예상과 차이 및 설명: **⟦직접 입력⟧**

## 3. 그래프와 오분류

실제 이미지와 예측을 함께 확인합니다.

### A. 기준 예제 · 먼저 읽고 실행

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(9,3));axes[0].plot(loss_history);axes[1].plot(accuracy_history)
for ax,yl in zip(axes,['train loss','train accuracy']):ax.set(xlabel='Epoch',ylabel=yl);ax.grid()
plt.show()
for title,items in [('test examples',examples),('misclassified',mistakes)]:
    if not items:print(title,': none in evaluated subset');continue
    fig,axes=plt.subplots(1,len(items),figsize=(2*len(items),2),squeeze=False)
    for ax,(image,truth,guess) in zip(axes[0],items):ax.imshow(image,cmap='gray');ax.set_title(f'T:{truth} P:{guess}');ax.axis('off')
    fig.suptitle(title);plt.show()



### B. 직접 수행

맞힌 예와 틀린 예를 보고 결과를 해석하세요. subset 결과와 전체 test accuracy를 혼동하지 않았는지 점검하세요.

먼저 결과를 예상하고 코드를 작성하세요. 위 예제의 결과만으로 이 항목을 완료한 것은 아닙니다.

In [ ]:
# ✍ 직접 작성: 이 셀 아래에 본인의 코드를 추가하세요.
# 코드가 길어지면 Code Cell을 추가해도 됩니다.


### C. 직접 기록 · 이 Text Cell을 더블클릭해서 수정

- 바꾼 조건: **⟦직접 입력⟧**
- 실행 전 예상: **⟦직접 입력⟧**
- 실제 출력/그래프에서 확인한 값: **⟦직접 입력⟧**
- 예상과 차이 및 설명: **⟦직접 입력⟧**

## 반복 숙달 · 실행 전에 판단하기

위에서 수행한 한 유형을 골라 입력 조건을 세 가지 더 바꾸세요. 한 번에 한 조건만 바꿉니다. 각 조건에서 예상 → 실행 → 확인을 반복합니다.

| 변경 조건 | 예상 shape/수치/결과 | 실제 결과 | 오류가 있었다면 원인 |
|---|---|---|---|
| ⟦직접 입력 1⟧ | ⟦직접 입력⟧ | ⟦직접 입력⟧ | ⟦직접 입력⟧ |
| ⟦직접 입력 2⟧ | ⟦직접 입력⟧ | ⟦직접 입력⟧ | ⟦직접 입력⟧ |
| ⟦직접 입력 3⟧ | ⟦직접 입력⟧ | ⟦직접 입력⟧ | ⟦직접 입력⟧ |

In [ ]:
# ✍ 반복 실험 코드


## 선택 실습 · 오류의 원인을 찾아 코드 고치기

앞 코드의 축·dtype·입력 크기·모델 설정 중 하나를 일부러 바꿔 예상과 달라지는 사례를 만드세요. 오류를 그대로 남기지 말고, 어떤 입력 조건이나 연산 규칙이 맞지 않았는지 설명한 뒤 수정된 코드로 실행하세요. 인증·설치 설정을 바꾸는 실험은 하지 않습니다.

## 저장 전 확인

1. 직접 작성란을 채우고 필요한 출력·그래프를 남겼는지 확인합니다.
2. 새 런타임에서 위에서부터 실행해 숨은 변수 의존성을 확인합니다.
3. 수정한 파일을 본인 Drive에 저장하고 `.ipynb`로 내려받습니다.
4. 제출 파일명·기한은 블로그의 이번 주 실습 안내와 최신 KLAS 공지를 따릅니다.